### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import sqlite3

GROUP = 'gp07'
DATA_URLS = {'measurement_areas.geojson': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp07/measurement_areas.geojson', 'measurement_routes.geojson': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp07/measurement_routes.geojson', 'measurement_sites.csv': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/gp07/measurement_sites.csv'}
SCHEMA = {'measurement_areas.geojson': {'kind': 'geojson', 'fields': ['area_id', 'name']}, 'measurement_routes.geojson': {'kind': 'geojson', 'fields': ['route_id', 'name']}, 'measurement_sites.csv': {'kind': 'csv', 'fields': ['site_id', 'name', 'longitude', 'latitude']}}

def valid_cached(path, spec):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if spec["kind"] == "csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                return set(spec["fields"]).issubset(set(reader.fieldnames or []))
        if spec["kind"] == "geojson":
            document = json.loads(path.read_text(encoding="utf-8"))
            if document.get("type") != "FeatureCollection" or not isinstance(document.get("features"), list):
                return False
            return all(isinstance(feature, dict) and isinstance(feature.get("properties"), dict) and set(spec["fields"]).issubset(feature["properties"]) for feature in document["features"])
        if spec["kind"] == "gpkg":
            if path.read_bytes()[:16] != b"SQLite format 3\x00":
                return False
            with sqlite3.connect(path) as connection:
                if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                    return False
                layers = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents")}
            return set(spec["layers"]).issubset(layers)
    except Exception:
        return False
    return False

def valid_dataset_root(base):
    folder = base / GROUP
    return folder.is_dir() and all(valid_cached(folder / filename, spec) for filename, spec in SCHEMA.items())

candidates = (Path("downloads/python/data"), Path("../../data"), Path("data"))
DATA_ROOT = next((candidate for candidate in candidates if valid_dataset_root(candidate)), None)

if DATA_ROOT is None:
    DATA_ROOT = Path("data")
    target_root = DATA_ROOT / GROUP
    target_root.mkdir(parents=True, exist_ok=True)
    failures = []
    for filename, url in DATA_URLS.items():
        target = target_root / filename
        if valid_cached(target, SCHEMA[filename]):
            print("[OK] Already available:", target)
            continue
        if target.exists():
            target.unlink()
        partial = target.with_name(target.name + ".part")
        if partial.exists():
            partial.unlink()
        try:
            with urlopen(url, timeout=30) as response:
                payload = response.read()
            if not payload:
                raise RuntimeError("empty response")
            partial.write_bytes(payload)
            partial.replace(target)
            if not valid_cached(target, SCHEMA[filename]):
                raise RuntimeError("minimum validation failed")
            print("[OK] Downloaded and validated:", target)
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if target.exists():
                target.unlink()
            failures.append(filename + ": " + str(exc))
    if failures or not valid_dataset_root(DATA_ROOT):
        raise RuntimeError("Could not prepare dataset " + GROUP + ": " + "; ".join(failures))
    print("[OK] Dataset root:", DATA_ROOT.resolve())
else:
    print("[OK] Using validated dataset root:", (DATA_ROOT / GROUP).resolve())


# GP07 — Measurements

Υπολόγισε μήκη, εμβαδά και αποστάσεις σε projected CRS. Μην κάνεις μετρήσεις σε μοίρες. Πριν από κάθε αποτέλεσμα έλεγξε geometry type, CRS και units. Correct code + wrong CRS = wrong measurement.

In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
root = DATA_ROOT / "gp07"
areas = gpd.read_file(root / 'measurement_areas.geojson')
routes = gpd.read_file(root / 'measurement_routes.geojson')
sites_df = pd.read_csv(root / 'measurement_sites.csv')
sites = gpd.GeoDataFrame(sites_df, geometry=gpd.points_from_xy(sites_df.longitude, sites_df.latitude), crs='EPSG:4326')


In [ ]:
target_crs = 'EPSG:____'
areas_m = areas.to_crs(target_crs)
routes_m = routes.to_crs(target_crs)
sites_m = sites.to_crs(target_crs)


In [ ]:
areas_m['area_m2'] = areas_m.geometry.area
areas_m['area_ha'] = areas_m['area_m2'] / ____
routes_m['length_m'] = routes_m.geometry.length
routes_m['length_km'] = routes_m['length_m'] / ____


In [ ]:
distance_m = sites_m.geometry.iloc[0].distance(sites_m.geometry.iloc[1])
print('distance:', distance_m, 'm')


QA: θετικά μεγέθη, μέτρα στο projected CRS, σωστό πλήθος features και λογικές τάξεις μεγέθους. Πρόσθεσε min/max/mean ή median όπου βοηθούν να βρεις outlier και εξήγησε γιατί η distance είναι γεωμετρική, όχι network distance.

In [ ]:
# assert ...
